# Week 5 Assignment: Deep Learning & Global Models
**DSE6230: Forecasting Methods and Applications**

In this assignment, you will explore **Global Models** using `neuralforecast`, then compare them against **zero-shot foundation models** that require no training at all.

1. **Data Prep**: Loading Panel Data (Long format).
2. **Model Training**: MLP / LSTM / NHITS / N-BEATS (local and global models).
3. **Zero-Shot Forecasting**: TimeGPT and/or Chronos.
4. **Synthesis**: Trained vs. zero-shot tradeoffs.

In [ ]:
!pip install pandas numpy matplotlib neuralforecast datasetsforecast nixtla chronos-forecasting torch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.utils import AirPassengersPanel

## Part 1: The Global Model
Use the `AirPassengers` panel data below, or an `M4` subset (10 series) if you prefer.

In [ ]:
# Raw data: long-format panel with columns unique_id, ds, y
df = AirPassengersPanel.copy()
horizon = 12

### Exercise 1.1: Load Data
Load a dataset containing at least 5 different time series. Ensure columns are `unique_id`, `ds`, `y`.

In [ ]:
# Your Code Here

### Exercise 1.2: Train NHITS
Initialize and fit an `NHITS` model with `input_size=2*horizon` and `h=horizon`.

In [ ]:
from neuralforecast.models import NHITS
from neuralforecast import NeuralForecast
# Your code

### Exercise 1.3: Compare to ARIMA
Why might NHITS outperform (or underperform) ARIMA on this specific dataset? Discuss "Transfer Learning" (information sharing across series).

### Exercise 1.4: The Gasoline Series (FPP Ch. 14, Q2)
Consider the weekly data on US finished motor gasoline products supplied (millions of barrels per day), series `us_gasoline`.

1. Use the `MLP` model to forecast the next 13 weeks of data.
2. Repeat using the `AutoMLP` and `AutoNHITS` models from `NeuralForecast`.
3. Compare the accuracy of the three models.

In [ ]:
from neuralforecast.models import MLP
from neuralforecast.auto import AutoMLP, AutoNHITS
from neuralforecast import NeuralForecast

# Your code

### Exercise 1.5: LSTM on the Pedestrian Dataset (FPP Ch. 14, Q1)
Use the `LSTM` model to forecast 24 steps ahead for **all** the series in the `pedestrian` dataset (hourly pedestrian counts across multiple Melbourne sensor locations — a natural panel/global-model dataset).

In [ ]:
from neuralforecast.models import LSTM
from neuralforecast import NeuralForecast

# Your code

### Exercise 1.6: Conceptual — Hidden Layers
Explain the role of "hidden layers" in a Multilayer Perceptron (MLP). How does increasing the number of layers affect model capacity and the risk of overfitting?

## Part 2: Zero-Shot Foundation Models
Unlike NHITS, which you just trained from scratch on your own panel, **time series foundation models** are pretrained on millions of series and can forecast new data with **no fitting step at all** — you just call `.forecast()`. Complete Exercise 2.1 (TimeGPT) **or** Exercise 2.2 (Chronos) — do both for extra credit.

### Exercise 2.1: TimeGPT (Nixtla)
`TimeGPT` is a hosted foundation model from Nixtla. It requires a free API key from [dashboard.nixtla.io](https://dashboard.nixtla.io) (sign up with a school or personal email).

In [ ]:
from nixtla import NixtlaClient

nixtla_client = NixtlaClient(api_key="YOUR_API_KEY_HERE")

# Reuse the same long-format DataFrame (unique_id, ds, y) from Part 1
fcst_df = nixtla_client.forecast(
    df=df,
    h=horizon,
    time_col="ds",
    target_col="y",
    id_col="unique_id",
    freq="M"  # match your data's frequency
)

1. Generate a forecast for the **same series and horizon** you used for NHITS in Exercise 1.2 (or, if you have Nixtla token access, use the `AirPassengers` dataset).
2. Plot the TimeGPT forecast against the actual holdout values using `nixtla_client.plot(...)`.
3. Compute a **Seasonal Naive** benchmark forecast for the same series/horizon and compare its error (e.g., MAE) against TimeGPT's. Did the zero-shot model beat the naive benchmark?
4. Note how long the forecast took to generate compared to training NHITS. What did you *not* have to do that you did for the global model?

### Exercise 2.2: Chronos (Amazon, fully local)
If you'd rather not create an API account, use **Chronos**, an open-source foundation model you can run locally with no key required.

In [ ]:
import torch
import pandas as pd
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.float32,
)

# Chronos forecasts one series at a time from a 1D context tensor
context = torch.tensor(df.loc[df["unique_id"] == "series_1", "y"].values)
quantiles, mean = pipeline.predict_quantiles(
    context=context,
    prediction_length=horizon,
    quantile_levels=[0.1, 0.5, 0.9],
)

1. Run Chronos on **one series** from your panel and plot the median forecast (`mean`) alongside the 10th/90th percentile band.
2. Loop over at least 3 series and record each model's error (e.g., MAE) in a small comparison table.
3. Chronos ships in multiple sizes (`tiny`, `mini`, `small`, `base`, `large`). Try swapping in a different size — does accuracy change? Does speed?

### Exercise 2.3: Conceptual — Zero-Shot vs. Transfer Learning
Describe the "zero-shot" inference capability of foundation models like TimeGPT. How does it differ from the "transfer learning" you discussed for NHITS in Exercise 1.3?

### Exercise 2.4: Synthesis — Trained vs. Zero-Shot
Using your results from Part 1 and Part 2, write a short (3-5 sentence) comparison addressing:

1. **Accuracy**: Which approach (NHITS, TimeGPT/Chronos, or ARIMA) performed best on your data, and why might that be?
2. **Cost of entry**: How much data, compute, and time did each approach require *before* it produced its first forecast?
3. **When to use which**: Describe a real business scenario where a zero-shot foundation model would be preferable to training a custom global model, and one where it wouldn't. What are the potential advantages of a foundation model (like Chronos or TimesFM) specifically when your dataset is *too small* to train a local or global model well?

## Grading Rubric
- **Correctness (35%)**: Does the DL pipeline (NHITS/MLP/LSTM) and at least one zero-shot model (TimeGPT or Chronos) run without errors?
- **Code Quality (25%)**: Proper data formatting (`unique_id`), reused consistently across models.
- **Zero-Shot Exercise (20%)**: Completed forecast + plot for TimeGPT or Chronos, with a basic accuracy comparison (including the Seasonal Naive benchmark).
- **Analysis (20%)**: Understanding of Local vs. Global modeling, and Trained vs. Zero-Shot tradeoffs (Exercises 2.3-2.4).